# imports

In [1]:
# !git clone https://github.com/moryacho/street-pattern-classifier.git

# !pip install huggingface_hub geopandas osmnx
# !pip install torch_geometric
# !pip install "folium>=0.12" matplotlib mapclassify

# !pip uninstall torch torchvision torchaudio -y
# !pip cache purge
# !pip install torch torchvision torchaudio

import torch

In [ ]:
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import time
from tqdm import tqdm
import os
import networkx as nx
import json

import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import warnings

from shapely.geometry import LineString, Polygon, Point
import shapely
import pickle


# read all polygons

big file (len = 85k) with polygions and data about travel mode choice

In [3]:
big_gdf_all_polygons = gpd.read_file("data//gdf_data_polygons_85k.geojson")

In [4]:
big_gdf_all_polygons.head(2)

,Geography,Geographic Area Name,Estimate!!Total:,"Estimate!!Total:!!Car, truck, or van:","Estimate!!Total:!!Car, truck, or van:!!Drove alone","Estimate!!Total:!!Car, truck, or van:!!Carpooled:","Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 2-person carpool","Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 3-person carpool","Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 4-or-more-person carpool",Estimate!!Total:!!Public transportation:,Estimate!!Total:!!Public transportation:!!Bus,Estimate!!Total:!!Public transportation:!!Subway or elevated rail,Estimate!!Total:!!Public transportation:!!Long-distance train or commuter rail,"Estimate!!Total:!!Public transportation:!!Light rail, streetcar or trolley (carro público in Puerto Rico)",Estimate!!Total:!!Public transportation:!!Ferryboat,Estimate!!Total:!!Bicycle,Estimate!!Total:!!Walked,"Estimate!!Total:!!Taxi or ride-hailing services, motorcycle, or other means",Estimate!!Total:!!Worked from home,geometry
0,1400000US01001020100,Census Tract 201; Autauga County; Alabama,803,787,691,96,96,0,0,0,0,0,0,0,0,0,0,0,16,"POLYGON ((-86.5091 32.47349, -86.50577 32.4757..."
1,1400000US01001020200,Census Tract 202; Autauga County; Alabama,977,763,762,1,1,0,0,0,0,0,0,0,0,0,5,0,209,"POLYGON ((-86.48122 32.47645, -86.48093 32.481..."


In [5]:
big_gdf_all_polygons.describe()

,Geography,Geographic Area Name,Estimate!!Total:,"Estimate!!Total:!!Car, truck, or van:","Estimate!!Total:!!Car, truck, or van:!!Drove alone","Estimate!!Total:!!Car, truck, or van:!!Carpooled:","Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 2-person carpool","Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 3-person carpool","Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 4-or-more-person carpool",Estimate!!Total:!!Public transportation:,Estimate!!Total:!!Public transportation:!!Bus,Estimate!!Total:!!Public transportation:!!Subway or elevated rail,Estimate!!Total:!!Public transportation:!!Long-distance train or commuter rail,"Estimate!!Total:!!Public transportation:!!Light rail, streetcar or trolley (carro público in Puerto Rico)",Estimate!!Total:!!Public transportation:!!Ferryboat,Estimate!!Total:!!Bicycle,Estimate!!Total:!!Walked,"Estimate!!Total:!!Taxi or ride-hailing services, motorcycle, or other means",Estimate!!Total:!!Worked from home,geometry
count,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378,84378
unique,84378,84378,5232,4325,3964,997,795,431,403,1462,694,1194,414,222,189,351,962,455,1777,84378
top,1400000US72153750602,Census Tract 7506.02; Yauco Municipio; Puerto ...,1488,1444,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"POLYGON ((-86.64561 32.555532, -86.643596 32.5..."
freq,1,1,60,67,72,1898,3342,35214,45105,41114,47783,73020,74512,80354,82509,63956,26436,26777,1310,1


# geo_place column

In [6]:
gdf_with_geo_place = big_gdf_all_polygons.copy()

gdf_with_geo_place['geo_place'] = gdf_with_geo_place['Geographic Area Name'].str.split(';').str[1] + gdf_with_geo_place['Geographic Area Name'].str.split(';').str[2]

# geo_place to osm_id

## funcs

In [7]:
def convert_numpy_types(obj):
    if isinstance(obj, dict):
        return {key: convert_numpy_types(value) for key, value in obj.items()}
    elif isinstance(obj, (np.int_, np.intc, np.intp, np.int8,
                          np.int16, np.int32, np.int64, np.uint8,
                          np.uint16, np.uint32, np.uint64)):
        return int(obj)
    elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.bool_)):
        return bool(obj)
    else:
        return obj

def save_dict_to_json(data_dict, filename='dictionary.json'):
    converted_dict = convert_numpy_types(data_dict)
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(converted_dict, f, ensure_ascii=False, indent=2)
    
    # print(f"✅ Словарь сохранен в {filename}")
    # print(f"   Количество элементов: {len(data_dict)}")

def load_dict_from_json(filename='dictionary.json'):

    with open(filename, 'r', encoding='utf-8') as f:
        data_dict = json.load(f)
    
    # print(f"✅ Словарь загружен из {filename}")
    # print(f"   Количество элементов: {len(data_dict)}")
    
    return data_dict

# save_dict_to_json(county_osm_dict, 'my_dict.json')
# county_osm_dict = load_dict_from_json('my_dict.json')

## usage

In [8]:
county_osm_dict = load_dict_from_json('my_dict.json')

In [9]:
len(county_osm_dict)

965

# read drive graphs dict by geo_places

### funcs

In [10]:
def save_graphs_to_pickle(graph_dict, filename='graphs_dictionary.pkl'):

    with open(filename, 'wb') as f:
        pickle.dump(graph_dict, f)
    
    # print(f"✅ Словарь сохранен в {filename}")
    # print(f"   Количество графов: {len(graph_dict)}")
    # print(f"   Размер файла: {os.path.getsize(filename) / (1024*1024):.2f} MB")

def load_graphs_from_pickle(filename='graphs_dictionary.pkl'):

    with open(filename, 'rb') as f:
        graph_dict = pickle.load(f)
    
    # print(f"✅ Словарь загружен из {filename}")
    # print(f"   Количество графов: {len(graph_dict)}")
    
    return graph_dict

# save_graphs_to_pickle(my_graph_dict, 'my_graphs.pkl')
# loaded_dict = load_graphs_from_pickle('my_graphs.pkl')

### usage

In [ ]:
# 3,7 gb file
# hard to open
# drive_graphs_dict = load_graphs_from_pickle('drive_graphs//drive_graphs_final.pkl') 

# mid dict
drive_graphs_dict = load_graphs_from_pickle('drive_graphs//drive_graphs_1.pkl') 

# if public transport graphs
# pt_graphs_dict = load_graphs_from_pickle('pt_graphs//pt_graphs_1.pkl') 


In [ ]:
len(drive_graphs_dict)

88

In [ ]:
# len(pt_graphs_dict)

# compare graph dict with gdf

## func

In [13]:
def add_graph_column(gdf : gpd.GeoDataFrame, 
                      graphs_dict : dict, 
                      graph_column : str = "graph",
                      target_crs : int = 32631) -> gpd.GeoDataFrame:

    gdf_list = [group for _, group in gdf.groupby('geo_place')]
    gdf_list_drive_graph = []

    for gdf_geo_place in tqdm(gdf_list):
        curr_geo_place = gdf_geo_place.iloc[0].geo_place
        if curr_geo_place not in graphs_dict:
            continue
        big_pt_g = nx.MultiDiGraph(graphs_dict[curr_geo_place])
        
        if big_pt_g.number_of_edges() == 0 and big_pt_g.number_of_nodes() == 0:
            continue
        
        g_crs = big_pt_g.graph["crs"] 
        gdf_geo_place_proj = gdf_geo_place.to_crs(g_crs)
        gdf_geo_place_proj[graph_column] = None
        
        nodes_gdf, edges_gdf = ox.graph_to_gdfs(nx.MultiDiGraph(big_pt_g))
        
        nodes_in_polygons = gpd.sjoin(nodes_gdf, gdf_geo_place_proj[['geometry']], 
                                    how='inner', predicate='within')
        nodes_by_polygon = nodes_in_polygons.groupby(nodes_in_polygons.index_right)
        
        new_nodes_to_add = {}
        
        for idx, row in gdf_geo_place_proj.iterrows():
            polygon_geom = row.geometry
            
            if idx in nodes_by_polygon.groups:
                polygon_nodes = nodes_by_polygon.get_group(idx)
                node_ids_inside = set(polygon_nodes.index)
            else:
                node_ids_inside = set()
            
            edges_to_add = []
            
            for u, v, key, data in big_pt_g.edges(keys=True, data=True):
                if u in node_ids_inside and v in node_ids_inside:
                    edges_to_add.append((u, v, key, data, 'full'))
                
                elif (u in node_ids_inside) != (v in node_ids_inside):  
                    if 'geometry' in data:
                        edge_geom = data['geometry']
                        
                        if edge_geom.intersects(polygon_geom):
                            intersection = edge_geom.intersection(polygon_geom)
                            
                            if not intersection.is_empty:
                                new_data = data.copy()
                                new_data['geometry'] = intersection
                                new_data['trimmed'] = True  
                                new_data['original_edge'] = (u, v, key)
                                
                                inside_node = u if u in node_ids_inside else v

                                if intersection.geom_type == 'LineString':
                                    boundary_point = intersection.coords[-1] if inside_node == u else intersection.coords[0]
                                elif intersection.geom_type == 'Point':
                                    boundary_point = intersection.coords[0]
                                else:
                                    continue
                                
                                boundary_node_id = f"boundary_{u}_{v}_{key}_{idx}"
                                
                                if boundary_node_id not in new_nodes_to_add and boundary_node_id not in big_pt_g.nodes:
                                    new_nodes_to_add[boundary_node_id] = {
                                        'x': boundary_point[0],
                                        'y': boundary_point[1],
                                        'boundary_node': True,
                                        'original_polygon': idx
                                    }
                                
                                if inside_node == u:
                                    edges_to_add.append((u, boundary_node_id, f"{key}_trimmed", new_data, 'trimmed'))
                                else:
                                    edges_to_add.append((boundary_node_id, v, f"{key}_trimmed", new_data, 'trimmed'))
                
                elif u not in node_ids_inside and v not in node_ids_inside:
                    if 'geometry' in data:
                        edge_geom = data['geometry']
                        
                        if edge_geom.intersects(polygon_geom):
                            intersection = edge_geom.intersection(polygon_geom)
                            
                            if not intersection.is_empty and intersection.geom_type == 'LineString':
                                coords = list(intersection.coords)
                                
                                if len(coords) >= 2:
                                    node_in_id = f"boundary_in_{u}_{v}_{key}_{idx}"
                                    node_out_id = f"boundary_out_{u}_{v}_{key}_{idx}"
                                    
                                    for node_id, point in [(node_in_id, coords[0]), (node_out_id, coords[-1])]:
                                        if node_id not in new_nodes_to_add and node_id not in big_pt_g.nodes:
                                            new_nodes_to_add[node_id] = {
                                                'x': point[0],
                                                'y': point[1],
                                                'boundary_node': True,
                                                'original_polygon': idx
                                            }
                                    
                                    new_data = data.copy()
                                    new_data['geometry'] = intersection
                                    new_data['trimmed'] = True
                                    new_data['original_edge'] = (u, v, key)
                                    
                                    edges_to_add.append((node_in_id, node_out_id, f"{key}_through", new_data, 'through'))
            
            if len(node_ids_inside) > 0 or len(edges_to_add) > 0:
                subgraph = nx.MultiDiGraph()
                subgraph.graph['crs'] = g_crs 
                subgraph.graph['parent_polygon'] = curr_geo_place
                
                for node_id in node_ids_inside:
                    node_data = big_pt_g.nodes[node_id]
                    subgraph.add_node(node_id, **node_data)
                
                for node_id, node_data in new_nodes_to_add.items():
                    if node_data.get('original_polygon') == idx:
                        subgraph.add_node(node_id, **node_data)
                
                for u, v, key, data, edge_type in edges_to_add:
                    subgraph.add_edge(u, v, key=key, **data)
                
                gdf_geo_place_proj.at[idx, graph_column] = subgraph
        
        gdf_final = gdf_geo_place_proj.to_crs(target_crs)
        
        for idx, row in gdf_final.iterrows():
            if row[graph_column] is not None:
                row[graph_column].graph['crs'] = str(target_crs)
        
        gdf_list_drive_graph.append(gdf_final)
    
    if gdf_list_drive_graph:
        return gpd.concat(gdf_list_drive_graph, ignore_index=True)
    else:
        return gdf.iloc[0:0]

## usage

In [17]:
gdf_with_graph_column = add_graph_column(gdf_with_geo_place, drive_graphs_dict)

 39%|███▊      | 1245/3222 [19:30<30:58,  1.06it/s] 


KeyboardInterrupt: 

# classification

## model

In [ ]:
from huggingface_hub import hf_hub_download
import torch

model_path = hf_hub_download(
    repo_id="nochka/street-pattern-classifier",
    filename="best_model.pth",
    local_dir="./models"
)

## prepare data

In [ ]:
def get_subgraphs_dict(gdf):
    subgraphs = {}
    for idx, row in gdf.iterrows():
        if row['graph'] is not None and row['graph'].number_of_edges() != 0:
            g = row['graph']
            G_copy = g.copy()
            all_geoms = []
            for u, v, key, data in G_copy.edges(keys=True, data=True):
                if 'geometry' not in data:
                    if ('x' in G_copy.nodes[u] and 'y' in G_copy.nodes[u] and 
                    'x' in G_copy.nodes[v] and 'y' in G_copy.nodes[v]):
                    
                        line = LineString([
                            (G_copy.nodes[u]['x'], G_copy.nodes[u]['y']),
                            (G_copy.nodes[v]['x'], G_copy.nodes[v]['y'])
                        ])
                        G_copy.edges[u, v, key]['geometry'] = line
                    else:
                        print('не получилось добавить геометрию ребра')
                else:
                    line = data['geometry']
                all_geoms.append(line)
            
            if all_geoms:
                all_geoms_union = shapely.union_all(all_geoms)
                bbox_geom = shapely.box(*all_geoms_union.bounds)

            subgraphs[idx] = {
                'graph': G_copy,
                # 'polygon': row['geometry']
                'polygon': bbox_geom
            }
    return subgraphs


In [ ]:
subgraphs = get_subgraphs_dict(gdf_with_drive_graphs)

## dataset

In [ ]:
import sys
sys.path.insert(0, './street-pattern-classifier')

from block_dataset import BlockDataset
dataset = BlockDataset(subgraphs)

In [ ]:
# torch.save(dataset, 'block_dataset_10k.pt')

# loaded_dataset = torch.load('block_dataset.pt')

## classify

In [ ]:
from classification import classify_blocks
predictions_blocks, probabilities_blocks = classify_blocks(
    dataset,
    model_path=model_path,
    device='cpu'
)

## get gdf with predictions

In [ ]:


def add_predictions_to_gdf(gdf, predictions_blocks, probabilities_blocks):

    gdf_with_predictions = gdf.copy()
    gdf_with_predictions['prediction'] = None
    
    num_classes = len(next(iter(probabilities_blocks.values()))) if probabilities_blocks else 7
    for i in range(num_classes):
        gdf_with_predictions[f'prob_{i}'] = None
    
    for idx in gdf_with_predictions.index:
        if idx in predictions_blocks:
            gdf_with_predictions.at[idx, 'prediction'] = predictions_blocks[idx]
            
            if idx in probabilities_blocks:
                probs = probabilities_blocks[idx]
                for i, prob in enumerate(probs):
                    gdf_with_predictions.at[idx, f'prob_{i}'] = prob
    
    return gdf_with_predictions


In [ ]:

gdf_with_predictions = add_predictions_to_gdf(
    gdf_with_drive_graphs, 
    predictions_blocks, 
    probabilities_blocks
)


print(gdf_with_predictions['prediction'].value_counts().sort_index())


## plot

### func

In [ ]:
warnings.filterwarnings('ignore')


def create_overlaid_kde_plots_with_stats(
    gdf,
    feature_columns,
    predictions_col='prediction',
    class_names=None,
    figsize=(12, 7),
    min_samples=5,
    lower_bound=None,
    upper_bound=None,
    lower_percentile=None,
    upper_percentile=None,
    bw_method='scott',
    n_points=400,
    stats_position=(0.98, 0.98),           # координаты блока со статистикой (в долях от осей)
    stats_fontsize=9.2,
    stats_alpha=0.92,
    stats_bbox=dict(facecolor='white', edgecolor='gray', alpha=0.85, boxstyle='round,pad=0.4')
):

    gdf_numeric = gdf.copy()
    for feature in feature_columns:
        if feature in gdf_numeric.columns:
            gdf_numeric[feature] = pd.to_numeric(gdf_numeric[feature], errors='coerce')

    classes = sorted(gdf_numeric[predictions_col].dropna().unique())
    n_classes = len(classes)

    colors = ['#FF8C00', '#1E88E5', '#DC143C', '#2E7D32', '#9467BD', '#FF69B4',
              '#8B4513', '#00CED1', '#FF1493', '#556B2F']

    if class_names is None:
        class_names = [f'Class {c}' for c in classes]

    all_filtered_data = {}
    bounds_info = {}

    for feature in feature_columns:
        feature_data = gdf_numeric[feature].dropna()
        if len(feature_data) < min_samples:
            print(f"Фича '{feature}' пропущена — слишком мало данных ({len(feature_data)})")
            continue

        # Определяем границы обрезки
        if lower_bound is not None and upper_bound is not None:
            lower_limit = lower_bound
            upper_limit = upper_bound
            bounds_info[feature] = f'[{lower_limit:.3f}, {upper_limit:.3f}] (fixed)'
        elif lower_percentile is not None and upper_percentile is not None:
            lower_limit = feature_data.quantile(lower_percentile)
            upper_limit = feature_data.quantile(upper_percentile)
            bounds_info[feature] = f'[{lower_percentile:.0%} – {upper_percentile:.0%}]'
        elif upper_percentile is not None:
            lower_limit = feature_data.min()
            upper_limit = feature_data.quantile(upper_percentile)
            bounds_info[feature] = f'[min – {upper_percentile:.0%}]'
        elif lower_percentile is not None:
            lower_limit = feature_data.quantile(lower_percentile)
            upper_limit = feature_data.max()
            bounds_info[feature] = f'[{lower_percentile:.0%} – max]'
        else:
            lower_limit = feature_data.min()
            upper_limit = feature_data.max()
            bounds_info[feature] = '[min, max]'

        filtered = feature_data[(feature_data >= lower_limit) & (feature_data <= upper_limit)]
        all_filtered_data[feature] = (filtered, lower_limit, upper_limit)

    for feature in feature_columns:
        if feature not in all_filtered_data:
            continue

        filtered_all, lower_limit, upper_limit = all_filtered_data[feature]
        x_range = upper_limit - lower_limit
        if x_range <= 0:
            print(f"Фича '{feature}' — нулевой диапазон после обрезки, пропускаем")
            continue

        x = np.linspace(lower_limit - x_range*0.04, upper_limit + x_range*0.04, n_points)

        fig, ax = plt.subplots(figsize=figsize)

        max_height = 0
        stats_lines = []

        for i, class_val in enumerate(classes):
            data = gdf_numeric[gdf_numeric[predictions_col] == class_val][feature].dropna()

            if len(data) < min_samples:
                continue

            data_trim = data[(data >= lower_limit) & (data <= upper_limit)]

            if len(data_trim) < min_samples:
                print(f"  → класс {class_val} ({class_names[i]}) после обрезки имеет {len(data_trim)} значений — пропущен")
                continue

            try:
                kde = stats.gaussian_kde(data_trim, bw_method=bw_method)
                density = kde(x)
                density_norm = density / density.max() * 0.92

                color = colors[i % len(colors)]

                ax.fill_between(x, 0, density_norm, color=color, alpha=0.28, linewidth=0)
                ax.plot(x, density_norm, color=color, lw=1.9, alpha=0.9, label=class_names[i])

                max_height = max(max_height, density_norm.max())

                # ─── Считаем статистику ───
                n = len(data_trim)
                mean_val = data_trim.mean()
                median_val = data_trim.median()
                std_val = data_trim.std()
                q25 = data_trim.quantile(0.25)
                q75 = data_trim.quantile(0.75)
                iqr = q75 - q25

                stats_lines.append(
                    f"{class_names[i]:<18} "
                    f"n={n:>4}  "
                    f"med={median_val:>6.2f}  "
                    f"mean={mean_val:>6.2f}  "
                    f"sd={std_val:>5.2f}  "
                    f"IQR={iqr:>5.2f}"
                )

            except Exception as e:
                print(f"KDE ошибка для '{feature}' / класс {class_val}: {e}")
                continue

        if max_height == 0:
            print(f"Фича '{feature}' — не удалось построить ни одну кривую")
            plt.close(fig)
            continue

        # ─── Оформление графика ───
        ax.set_xlim(x[0], x[-1])
        ax.set_ylim(0, max_height * 1.15)

        ax.set_xlabel('Value', fontsize=11)
        ax.set_ylabel('Normalized density', fontsize=11)

        short_name = feature.replace('Estimate!!Total:', '').replace('Estimate!!', '').strip()
        if len(short_name) > 60:
            short_name = short_name[:57] + '...'

        ax.set_title(f"{short_name}\nby Urban Street Pattern Typology", 
                     fontsize=13, fontweight='bold', pad=14)

        if feature in bounds_info:
            ax.text(0.02, 0.96, f"bounds: {bounds_info[feature]}",
                    transform=ax.transAxes, ha='left', va='top',
                    fontsize=9, color='gray', alpha=0.8)

        ax.legend(loc='upper right', fontsize=9.5, framealpha=0.92,
                  edgecolor='gray', bbox_to_anchor=(0.99, 0.78))

        # ─── Вывод статистики в виде текста ───
        if stats_lines:
            stats_text = "Statistics (after trimming):\n" + "\n".join(stats_lines)
            ax.text(
                stats_position[0], stats_position[1],
                stats_text,
                transform=ax.transAxes,
                fontsize=stats_fontsize,
                fontfamily='monospace',
                ha='right',
                va='top',
                bbox=stats_bbox,
                alpha=stats_alpha
            )

        ax.grid(True, axis='y', alpha=0.25, linestyle='--', linewidth=0.8)
        ax.tick_params(labelsize=9.5)

        plt.tight_layout()
        plt.show()

### usage

In [ ]:
feature_columns = [
    # 'Estimate!!Total:',
    'Estimate!!Total:!!Car, truck, or van:',
    'Estimate!!Total:!!Car, truck, or van:!!Drove alone',
    'Estimate!!Total:!!Car, truck, or van:!!Carpooled:',
    'Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 2-person carpool',
    'Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 3-person carpool',
    'Estimate!!Total:!!Car, truck, or van:!!Carpooled:!!In 4-or-more-person carpool',
    'Estimate!!Total:!!Public transportation:',
    'Estimate!!Total:!!Public transportation:!!Bus',
    'Estimate!!Total:!!Public transportation:!!Subway or elevated rail',
    'Estimate!!Total:!!Public transportation:!!Long-distance train or commuter rail',
    'Estimate!!Total:!!Public transportation:!!Light rail, streetcar or trolley (carro público in Puerto Rico)',
    'Estimate!!Total:!!Public transportation:!!Ferryboat',
    'Estimate!!Total:!!Bicycle',
    'Estimate!!Total:!!Walked',
    'Estimate!!Total:!!Taxi or ride-hailing services, motorcycle, or other means',
    'Estimate!!Total:!!Worked from home',
    # 'area'
]

class_names = ["Loops & Lollipops", "Irregular Grid", "Regular Grid", "Warped Parallel", "Sparse", "Broken Grid"]



create_overlaid_kde_plots_with_stats(
    gdf_with_predictions,
    feature_columns,
    class_names=class_names,
    figsize=(12, 6),
    min_samples=5,
    lower_percentile=0.05,
    upper_percentile=0.95,
    bw_method=0.3,
    stats_position=(0.98, 0.98),          # можно сдвинуть, напр. (0.98, 0.65)
    stats_fontsize=9.0,
)